# 11 — Dual-Stream Face + Mouth V4 Stable Training (100 epochs)

เวอร์ชันนี้เขียนใหม่เพื่อแก้จุดเสี่ยงจาก V3 โดยเน้น **ความเสถียร + ตรวจข้อมูลจริงก่อน CTC**

จุดสำคัญของ V4:
- ใช้ Face + Mouth แบบ dual-stream
- สร้าง fast cache ต่อคลิป (`.npz`) ก่อนเทรน
- **ตรวจ CTC compatibility จากจำนวนเฟรมจริงหลัง cache** ไม่ใช่จากจำนวน path ใน CSV
- ตัด sample ที่ `T < minimum CTC steps` ออกอย่างชัดเจน
- ใช้ mild anti-blank regularizer เฉพาะช่วงต้น และค่อย ๆ ลดเป็น 0
- AMP + gradient accumulation สำหรับ RTX 4050 6GB
- เก็บ `best_model.pt`, `last_model.pt`, history CSV
- รองรับ 100 epochs + early stopping แบบไม่หยุดเร็วเกินไป

> ให้รันจาก Cell 1 ลงมาตามลำดับ อย่าข้าม Cell ที่สร้าง cache / ตรวจข้อมูล


In [ ]:
# CELL 1 — Kernel / Python test
import sys
print('='*80)
print('KERNEL TEST')
print('='*80)
print('Python executable :', sys.executable)
print('Python version    :', sys.version.split()[0])
print('KERNEL OK')


In [ ]:
# CELL 2 — Imports + CUDA test
import os
import json
import math
import random
import re
import time
import unicodedata
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

print('='*80)
print('ENVIRONMENT')
print('='*80)
print('Torch :', torch.__version__)
print('CUDA  :', torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError('CUDA ไม่พร้อมใช้งาน — หยุดตรงนี้ก่อน')

DEVICE = torch.device('cuda:0')
print('GPU   :', torch.cuda.get_device_name(0))
print('VRAM  :', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), 'GB')

torch.backends.cudnn.benchmark = True
try:
    torch.set_float32_matmul_precision('high')
except Exception:
    pass


In [ ]:
# CELL 3 — CONFIG
# แก้ค่าหลักของการเทรนที่ Cell นี้เท่านั้น

PROJECT_ROOT = Path(r'D:\Face_detection')
VIDEO_ID = 'video001'

SPLIT_ROOT = PROJECT_ROOT / 'dataset' / 'splits' / VIDEO_ID
TRAIN_CSV = SPLIT_ROOT / 'train.csv'
VAL_CSV   = SPLIT_ROOT / 'val.csv'
TEST_CSV  = SPLIT_ROOT / 'test.csv'

# face crops ที่สร้างจากขั้นตอน face detection / re-id เดิม
FACE_CACHE_ROOT = PROJECT_ROOT / 'dataset' / 'face_crops_v2' / VIDEO_ID

# V4 จะสร้าง cache ใหม่ของทั้ง face + mouth ต่อคลิป
FAST_CACHE_ROOT = PROJECT_ROOT / 'dataset' / 'dualstream_fast_cache_v4' / VIDEO_ID
MODEL_ROOT = PROJECT_ROOT / 'models' / 'lipreading_dualstream_v4'

FAST_CACHE_ROOT.mkdir(parents=True, exist_ok=True)
MODEL_ROOT.mkdir(parents=True, exist_ok=True)

# ---------- Data ----------
FACE_SIZE = 112
MOUTH_SIZE = 96
MAX_FRAMES = 180
BATCH_SIZE = 2
NUM_WORKERS = 0      # Windows + Jupyter: ใช้ 0 เพื่อความเสถียร
GRAD_ACCUM_STEPS = 4 # effective batch ~= 8

# ---------- Model ----------
FEATURE_DIM = 128
GRU_HIDDEN = 192
GRU_LAYERS = 2
DROPOUT = 0.10

# ---------- Training ----------
EPOCHS = 100
LR = 2e-4
WEIGHT_DECAY = 1e-4
GRAD_CLIP = 5.0
PATIENCE = 20
MIN_EPOCHS_BEFORE_EARLY_STOP = 15
SEED = 42

# mild anti-blank: ใช้เฉพาะช่วงแรก ไม่บังคับตลอดการเทรน
ANTI_BLANK_WEIGHT = 0.05
ANTI_BLANK_MAX_EPOCH = 20
ANTI_BLANK_THRESHOLD = 0.95

# ถ้าต้องการเริ่มรอบใหม่จริง ๆ ให้ False
RESUME_IF_AVAILABLE = False

print('='*80)
print('PATH CHECK')
print('='*80)
for p in [TRAIN_CSV, VAL_CSV, TEST_CSV, FACE_CACHE_ROOT]:
    print(f'{p} -> {p.exists()}')
print('FAST_CACHE_ROOT:', FAST_CACHE_ROOT)
print('MODEL_ROOT     :', MODEL_ROOT)


In [ ]:
# CELL 4 — Seed
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
print('Seed ready:', SEED)


In [ ]:
# CELL 5 — Load split CSVs + validate columns
train_df = pd.read_csv(TRAIN_CSV)
val_df   = pd.read_csv(VAL_CSV)
test_df  = pd.read_csv(TEST_CSV)

REQUIRED_COLUMNS = {'clip_id', 'speaker_id', 'text', 'frame_paths'}

for name, df in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    missing = REQUIRED_COLUMNS - set(df.columns)
    if missing:
        raise KeyError(f'{name} CSV ขาด columns: {sorted(missing)}')
    print(f'{name:5s}: {len(df)} clips')

print()
print('Train columns:', train_df.columns.tolist())
display(train_df.head(3))


In [ ]:
# CELL 6 — Text normalization + tokenizer

def normalize_text(text):
    text = unicodedata.normalize('NFC', str(text))
    text = ' '.join(text.split()).strip()
    return text

for df in (train_df, val_df, test_df):
    df['text'] = df['text'].fillna('').astype(str).map(normalize_text)

BLANK_TOKEN = '<blank>'
UNK_TOKEN = '<unk>'

# สร้าง vocabulary จาก train เท่านั้น ป้องกัน data leakage
charset = sorted(set(''.join(train_df['text'].tolist())))
charset = [c for c in charset if ord(c) not in (10, 13, 9)]

itos = [BLANK_TOKEN, UNK_TOKEN] + charset
stoi = {token: idx for idx, token in enumerate(itos)}

BLANK_ID = stoi[BLANK_TOKEN]
UNK_ID = stoi[UNK_TOKEN]
VOCAB_SIZE = len(itos)


def encode_text(text):
    return [stoi.get(ch, UNK_ID) for ch in normalize_text(text)]


def decode_ids(ids):
    chars = []
    for idx in ids:
        idx = int(idx)
        if idx == BLANK_ID:
            continue
        token = itos[idx]
        chars.append('�' if token == UNK_TOKEN else token)
    return ''.join(chars)


def ctc_greedy_decode(logits, lengths):
    pred_ids = logits.argmax(dim=-1)
    outputs = []
    for b in range(pred_ids.size(0)):
        T = int(lengths[b])
        seq = pred_ids[b, :T].tolist()
        collapsed = []
        prev = None
        for idx in seq:
            if idx != prev and idx != BLANK_ID:
                collapsed.append(idx)
            prev = idx
        outputs.append(decode_ids(collapsed))
    return outputs

with open(MODEL_ROOT / 'charset.json', 'w', encoding='utf-8') as f:
    json.dump({'itos': itos, 'blank_id': BLANK_ID, 'unk_id': UNK_ID}, f, ensure_ascii=False, indent=2)

print('VOCAB_SIZE:', VOCAB_SIZE)
print('BLANK_ID  :', BLANK_ID)
print('UNK_ID    :', UNK_ID)
print('Sample chars:', itos[:20])


In [ ]:
# CELL 7 — Helpers สำหรับ path / CTC minimum steps
frame_re = re.compile(r'frame_(\d+)', re.IGNORECASE)


def resolve_path(p):
    p = Path(str(p))
    return p if p.is_absolute() else PROJECT_ROOT / p


def parse_frame_paths(value):
    return [p.strip() for p in str(value).split('|') if p.strip()]


def face_cache_path(speaker_id, frame_index):
    return (
        FACE_CACHE_ROOT
        / f'speaker_{int(speaker_id):03d}'
        / f'frame_{int(frame_index):08d}.jpg'
    )


def safe_clip_name(clip_id):
    return re.sub(r'[^A-Za-z0-9_.-]+', '_', str(clip_id))


def clip_cache_path(clip_id):
    return FAST_CACHE_ROOT / f'{safe_clip_name(clip_id)}.npz'


def min_ctc_steps_from_ids(ids):
    # CTC ต้องมีเวลาเพิ่มอีก 1 step เมื่อ target มีตัวซ้ำติดกัน เช่น กก
    if len(ids) == 0:
        return 0
    repeats = sum(ids[i] == ids[i-1] for i in range(1, len(ids)))
    return len(ids) + repeats


def min_ctc_steps(text):
    return min_ctc_steps_from_ids(encode_text(text))

print('Helpers ready')


In [ ]:
# CELL 8 — Build FAST CACHE V4
# สำคัญ: V4 จะตรวจคู่ face/mouth ที่มีอยู่จริงก่อนเลือก MAX_FRAMES
# แล้วค่อยบันทึกเป็น .npz ต่อคลิป ลดการเปิด JPG หลายร้อยไฟล์ต่อ batch


def evenly_select(items, max_items):
    if len(items) <= max_items:
        return items
    indices = np.linspace(0, len(items) - 1, max_items).round().astype(int)
    return [items[i] for i in indices]


def build_one_cache(row, overwrite=False):
    dst = clip_cache_path(row['clip_id'])
    if dst.exists() and not overwrite:
        try:
            with np.load(dst, allow_pickle=False) as z:
                t = int(z['face'].shape[0])
                if t > 0 and z['mouth'].shape[0] == t:
                    return {'status': 'skip', 'T': t, 'cache': str(dst)}
        except Exception:
            pass

    speaker_id = int(row['speaker_id'])
    mouth_paths = parse_frame_paths(row['frame_paths'])

    # 1) หาคู่ face/mouth ที่ path มีอยู่จริง
    pairs = []
    for mouth_path_raw in mouth_paths:
        mouth_path = resolve_path(mouth_path_raw)
        m = frame_re.search(mouth_path.stem)
        if m is None:
            continue
        frame_index = int(m.group(1))
        face_path = face_cache_path(speaker_id, frame_index)
        if mouth_path.exists() and face_path.exists():
            pairs.append((frame_index, face_path, mouth_path))

    # 2) ค่อย downsample หลังรู้จำนวนคู่จริง
    pairs = evenly_select(pairs, MAX_FRAMES)

    faces = []
    mouths = []
    frame_indices = []

    for frame_index, face_path, mouth_path in pairs:
        face = cv2.imread(str(face_path), cv2.IMREAD_GRAYSCALE)
        mouth = cv2.imread(str(mouth_path), cv2.IMREAD_GRAYSCALE)

        if face is None or mouth is None:
            continue

        if face.shape != (FACE_SIZE, FACE_SIZE):
            face = cv2.resize(face, (FACE_SIZE, FACE_SIZE), interpolation=cv2.INTER_AREA)
        if mouth.shape != (MOUTH_SIZE, MOUTH_SIZE):
            mouth = cv2.resize(mouth, (MOUTH_SIZE, MOUTH_SIZE), interpolation=cv2.INTER_AREA)

        faces.append(face.astype(np.uint8))
        mouths.append(mouth.astype(np.uint8))
        frame_indices.append(frame_index)

    if len(faces) == 0:
        return {'status': 'empty', 'T': 0, 'cache': str(dst)}

    face_arr = np.stack(faces, axis=0)
    mouth_arr = np.stack(mouths, axis=0)
    frame_idx_arr = np.asarray(frame_indices, dtype=np.int32)

    # np.savez = ไม่บีบอัด เน้นโหลดเร็วกว่า np.savez_compressed
    np.savez(dst, face=face_arr, mouth=mouth_arr, frame_indices=frame_idx_arr)
    return {'status': 'built', 'T': len(face_arr), 'cache': str(dst)}


def build_cache_for_df(df, split_name, overwrite=False):
    rows = []
    t0 = time.time()
    for i, (_, row) in enumerate(df.iterrows(), start=1):
        info = build_one_cache(row, overwrite=overwrite)
        rows.append({
            'clip_id': row['clip_id'],
            'split': split_name,
            'status': info['status'],
            'T': info['T'],
            'cache': info['cache'],
        })
        if i == 1 or i % 50 == 0 or i == len(df):
            print(f'{split_name:5s} {i:4d}/{len(df)} | last={info["status"]} | T={info["T"]}')
    print(f'{split_name} cache finished in {(time.time()-t0)/60:.1f} min')
    return pd.DataFrame(rows)


train_cache_manifest = build_cache_for_df(train_df, 'train', overwrite=False)
val_cache_manifest   = build_cache_for_df(val_df, 'val', overwrite=False)
test_cache_manifest  = build_cache_for_df(test_df, 'test', overwrite=False)

cache_manifest = pd.concat(
    [train_cache_manifest, val_cache_manifest, test_cache_manifest],
    ignore_index=True
)
cache_manifest.to_csv(FAST_CACHE_ROOT / 'cache_manifest.csv', index=False, encoding='utf-8-sig')

print()
print('Cache status:')
display(cache_manifest.groupby(['split', 'status']).size().reset_index(name='count'))


In [ ]:
# CELL 9 — ตรวจ CTC compatibility จากจำนวนเฟรมจริงใน cache
# นี่คือจุดที่ต่างจาก V3 อย่างสำคัญ


def attach_actual_T(df):
    out = df.copy().reset_index(drop=True)
    actual_T = []
    for _, row in out.iterrows():
        cp = clip_cache_path(row['clip_id'])
        if not cp.exists():
            actual_T.append(0)
            continue
        try:
            with np.load(cp, allow_pickle=False) as z:
                t_face = int(z['face'].shape[0])
                t_mouth = int(z['mouth'].shape[0])
                actual_T.append(t_face if t_face == t_mouth else 0)
        except Exception:
            actual_T.append(0)
    out['actual_T'] = actual_T
    out['target_len'] = out['text'].map(lambda x: len(encode_text(x)))
    out['ctc_min_steps'] = out['text'].map(min_ctc_steps)
    out['ctc_valid'] = (
        (out['actual_T'] > 0)
        & (out['target_len'] > 0)
        & (out['actual_T'] >= out['ctc_min_steps'])
    )
    return out


def filter_valid(df, name):
    checked = attach_actual_T(df)
    bad = checked[~checked['ctc_valid']].copy()
    good = checked[checked['ctc_valid']].copy().reset_index(drop=True)

    print('='*80)
    print(name)
    print('='*80)
    print('Before :', len(checked))
    print('Valid  :', len(good))
    print('Removed:', len(bad))
    if len(bad):
        display(bad[['clip_id','actual_T','target_len','ctc_min_steps','text']].head(10))
    return good

train_df = filter_valid(train_df, 'TRAIN CTC CHECK')
val_df   = filter_valid(val_df, 'VAL CTC CHECK')
test_df  = filter_valid(test_df, 'TEST CTC CHECK')

print()
print('Final sizes:', len(train_df), len(val_df), len(test_df))


In [ ]:
# CELL 10 — Dataset + Collate
class FastDualStreamDataset(Dataset):
    def __init__(self, dataframe, training=False):
        self.df = dataframe.reset_index(drop=True).copy()
        self.training = training

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        cp = clip_cache_path(row['clip_id'])

        with np.load(cp, allow_pickle=False) as z:
            face = z['face'].copy()
            mouth = z['mouth'].copy()

        if face.shape[0] != mouth.shape[0]:
            raise RuntimeError(f'Face/Mouth length mismatch: {row["clip_id"]}')

        target_ids = encode_text(row['text'])
        need = min_ctc_steps_from_ids(target_ids)
        if face.shape[0] < need:
            raise RuntimeError(
                f'CTC invalid after cache: {row["clip_id"]} | T={face.shape[0]} need={need}'
            )

        # augmentation เบา ๆ เฉพาะ train
        if self.training and random.random() < 0.30:
            alpha = random.uniform(0.95, 1.05)
            beta = random.uniform(-5, 5)
            face = np.clip(face.astype(np.float32) * alpha + beta, 0, 255).astype(np.uint8)
            mouth = np.clip(mouth.astype(np.float32) * alpha + beta, 0, 255).astype(np.uint8)

        face = torch.from_numpy(face.astype(np.float32) / 255.0).unsqueeze(1)
        mouth = torch.from_numpy(mouth.astype(np.float32) / 255.0).unsqueeze(1)
        target = torch.tensor(target_ids, dtype=torch.long)

        return {
            'face': face,
            'mouth': mouth,
            'target': target,
            'text': row['text'],
            'clip_id': row['clip_id'],
        }


def collate_fn(batch):
    batch_size = len(batch)
    video_lengths = torch.tensor([x['face'].shape[0] for x in batch], dtype=torch.long)
    target_lengths = torch.tensor([x['target'].numel() for x in batch], dtype=torch.long)

    max_T = int(video_lengths.max())

    face_batch = torch.zeros(batch_size, max_T, 1, FACE_SIZE, FACE_SIZE, dtype=torch.float32)
    mouth_batch = torch.zeros(batch_size, max_T, 1, MOUTH_SIZE, MOUTH_SIZE, dtype=torch.float32)

    for i, x in enumerate(batch):
        T = x['face'].shape[0]
        face_batch[i, :T] = x['face']
        mouth_batch[i, :T] = x['mouth']

    targets = torch.cat([x['target'] for x in batch], dim=0)

    return {
        'face': face_batch,
        'mouth': mouth_batch,
        'video_lengths': video_lengths,
        'targets': targets,
        'target_lengths': target_lengths,
        'texts': [x['text'] for x in batch],
        'clip_ids': [x['clip_id'] for x in batch],
    }

print('Dataset / collate ready')


In [ ]:
# CELL 11 — DataLoaders + speed check
train_dataset = FastDualStreamDataset(train_df, training=True)
val_dataset   = FastDualStreamDataset(val_df, training=False)
test_dataset  = FastDualStreamDataset(test_df, training=False)

loader_common = dict(
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    collate_fn=collate_fn,
)

train_loader = DataLoader(train_dataset, shuffle=True, **loader_common)
val_loader   = DataLoader(val_dataset, shuffle=False, **loader_common)
test_loader  = DataLoader(test_dataset, shuffle=False, **loader_common)

print('Train batches:', len(train_loader))
print('Val batches  :', len(val_loader))
print('Test batches :', len(test_loader))

t0 = time.time()
batch = next(iter(train_loader))
print('First batch load:', round(time.time() - t0, 2), 'sec')
print('Face :', batch['face'].shape)
print('Mouth:', batch['mouth'].shape)
print('Video lengths :', batch['video_lengths'].tolist())
print('Target lengths:', batch['target_lengths'].tolist())
print('Texts:', batch['texts'])


In [ ]:
# CELL 12 — Model V4
class VisualEncoderV4(nn.Module):
    def __init__(self, out_dim=128):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 24, kernel_size=5, padding=2, bias=False),
            nn.GroupNorm(6, 24),
            nn.SiLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(24, 48, kernel_size=3, padding=1, bias=False),
            nn.GroupNorm(8, 48),
            nn.SiLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(48, 96, kernel_size=3, padding=1, bias=False),
            nn.GroupNorm(12, 96),
            nn.SiLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.proj = nn.Sequential(
            nn.Flatten(),
            nn.Linear(96, out_dim),
            nn.LayerNorm(out_dim),
        )

    def forward(self, x):
        return self.proj(self.features(x))


class DualStreamLipReadingV4(nn.Module):
    def __init__(self, vocab_size, feature_dim=128, hidden_dim=192, gru_layers=2, dropout=0.10):
        super().__init__()
        self.face_encoder = VisualEncoderV4(feature_dim)
        self.mouth_encoder = VisualEncoderV4(feature_dim)

        # learned gate: ให้โมเดลเลือกว่า timestep นี้ควรพึ่ง mouth หรือ face มากเท่าไร
        self.gate = nn.Sequential(
            nn.Linear(feature_dim * 2, feature_dim),
            nn.Sigmoid(),
        )
        self.fusion_norm = nn.LayerNorm(feature_dim)

        self.temporal = nn.GRU(
            input_size=feature_dim,
            hidden_size=hidden_dim,
            num_layers=gru_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if gru_layers > 1 else 0.0,
        )

        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_dim * 2, vocab_size)

        # เริ่มต้นโดยลดความได้เปรียบของ blank เล็กน้อย
        with torch.no_grad():
            self.classifier.bias.zero_()
            self.classifier.bias[BLANK_ID] = -0.75

    def forward(self, face, mouth, lengths):
        B, T = face.shape[:2]

        face_flat = face.reshape(B * T, 1, FACE_SIZE, FACE_SIZE)
        mouth_flat = mouth.reshape(B * T, 1, MOUTH_SIZE, MOUTH_SIZE)

        f = self.face_encoder(face_flat)
        m = self.mouth_encoder(mouth_flat)

        g = self.gate(torch.cat([f, m], dim=-1))
        fused = g * m + (1.0 - g) * f
        fused = self.fusion_norm(fused).reshape(B, T, -1)

        packed = nn.utils.rnn.pack_padded_sequence(
            fused,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False,
        )
        packed_out, _ = self.temporal(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(
            packed_out,
            batch_first=True,
            total_length=T,
        )

        logits = self.classifier(self.dropout(out))
        return logits


model = DualStreamLipReadingV4(
    vocab_size=VOCAB_SIZE,
    feature_dim=FEATURE_DIM,
    hidden_dim=GRU_HIDDEN,
    gru_layers=GRU_LAYERS,
    dropout=DROPOUT,
).to(DEVICE)

num_params = sum(p.numel() for p in model.parameters())
print(model)
print(f'Parameters: {num_params:,}')
print('Device:', DEVICE)


In [ ]:
# CELL 13 — Loss / optimizer / scheduler / mild anti-blank
criterion = nn.CTCLoss(
    blank=BLANK_ID,
    reduction='mean',
    zero_infinity=True,
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=3,
    min_lr=1e-6,
)

scaler = torch.amp.GradScaler('cuda')


def current_anti_blank_weight(epoch):
    if epoch > ANTI_BLANK_MAX_EPOCH:
        return 0.0
    progress = (epoch - 1) / max(ANTI_BLANK_MAX_EPOCH - 1, 1)
    return ANTI_BLANK_WEIGHT * (1.0 - progress)


def anti_blank_loss(logits, lengths):
    # ลงโทษเฉพาะกรณี probability ของ blank สูงเกิน 0.95 โดยเฉลี่ย
    # ไม่บังคับให้ blank ต่ำตลอด เพราะ CTC จำเป็นต้องใช้ blank
    probs = logits.float().softmax(dim=-1)
    blank_probs = probs[..., BLANK_ID]

    penalties = []
    for b in range(logits.size(0)):
        T = int(lengths[b])
        if T <= 0:
            continue
        mean_blank = blank_probs[b, :T].mean()
        penalties.append(F.relu(mean_blank - ANTI_BLANK_THRESHOLD).pow(2))

    if not penalties:
        return logits.new_tensor(0.0)
    return torch.stack(penalties).mean()

print('Loss/optimizer ready')


In [ ]:
# CELL 14 — Metrics

def edit_distance(a, b):
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, start=1):
        cur = [i]
        for j, cb in enumerate(b, start=1):
            cur.append(min(
                cur[-1] + 1,
                prev[j] + 1,
                prev[j-1] + (ca != cb),
            ))
        prev = cur
    return prev[-1]


def corpus_cer(refs, preds):
    total_edits = sum(edit_distance(r, p) for r, p in zip(refs, preds))
    total_chars = sum(len(r) for r in refs)
    return total_edits / max(total_chars, 1)


def argmax_blank_ratio(logits, lengths):
    pred = logits.argmax(dim=-1)
    blank = 0
    total = 0
    for b in range(pred.size(0)):
        T = int(lengths[b])
        seq = pred[b, :T]
        blank += int((seq == BLANK_ID).sum().item())
        total += T
    return blank / max(total, 1)

print('Metrics ready')


In [ ]:
# CELL 15 — REAL PREFLIGHT: forward + CTC + backward 1 batch
print('='*80)
print('REAL PREFLIGHT')
print('='*80)

batch = next(iter(train_loader))
face = batch['face'].to(DEVICE, non_blocking=True)
mouth = batch['mouth'].to(DEVICE, non_blocking=True)
lengths = batch['video_lengths']
targets = batch['targets'].to(DEVICE, non_blocking=True)
target_lengths = batch['target_lengths']

optimizer.zero_grad(set_to_none=True)

with torch.amp.autocast('cuda', dtype=torch.float16):
    logits = model(face, mouth, lengths)
    log_probs = logits.log_softmax(dim=-1).transpose(0, 1)
    ctc = criterion(log_probs, targets, lengths, target_lengths)
    ab = anti_blank_loss(logits, lengths)
    total = ctc + current_anti_blank_weight(1) * ab

print('Face          :', face.shape)
print('Mouth         :', mouth.shape)
print('Logits        :', logits.shape)
print('CTC loss      :', float(ctc.detach().cpu()))
print('Anti-blank    :', float(ab.detach().cpu()))
print('Total loss    :', float(total.detach().cpu()))
print('Blank ratio   :', argmax_blank_ratio(logits.detach(), lengths))

if not torch.isfinite(total):
    raise RuntimeError('Loss is NaN/Inf — หยุดก่อนเทรนจริง')

scaler.scale(total).backward()
scaler.unscale_(optimizer)
grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
optimizer.zero_grad(set_to_none=True)

print('Grad norm     :', float(grad_norm))
print('✅ REAL PREFLIGHT PASS')


In [ ]:
# CELL 16 — Train / Evaluate functions

def train_one_epoch(model, loader, epoch):
    model.train()
    optimizer.zero_grad(set_to_none=True)

    losses = []
    ctc_losses = []
    anti_losses = []
    start = time.time()
    anti_w = current_anti_blank_weight(epoch)

    for step, batch in enumerate(loader, start=1):
        face = batch['face'].to(DEVICE, non_blocking=True)
        mouth = batch['mouth'].to(DEVICE, non_blocking=True)
        lengths = batch['video_lengths']
        targets = batch['targets'].to(DEVICE, non_blocking=True)
        target_lengths = batch['target_lengths']

        with torch.amp.autocast('cuda', dtype=torch.float16):
            logits = model(face, mouth, lengths)
            log_probs = logits.log_softmax(dim=-1).transpose(0, 1)

            ctc = criterion(
                log_probs,
                targets,
                lengths,
                target_lengths,
            )
            anti = anti_blank_loss(logits, lengths)
            loss = ctc + anti_w * anti
            scaled_loss = loss / GRAD_ACCUM_STEPS

        if not torch.isfinite(loss):
            raise RuntimeError(f'NaN/Inf loss at epoch={epoch}, step={step}')

        scaler.scale(scaled_loss).backward()

        if step % GRAD_ACCUM_STEPS == 0 or step == len(loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        losses.append(float(loss.detach().cpu()))
        ctc_losses.append(float(ctc.detach().cpu()))
        anti_losses.append(float(anti.detach().cpu()))

        if step == 1 or step % 10 == 0 or step == len(loader):
            elapsed = time.time() - start
            sec_per_step = elapsed / step
            eta_min = sec_per_step * (len(loader) - step) / 60
            print(
                f'train {step:4d}/{len(loader)}'
                f' | loss={np.mean(losses):.4f}'
                f' | ctc={np.mean(ctc_losses):.4f}'
                f' | anti={np.mean(anti_losses):.6f}'
                f' | anti_w={anti_w:.4f}'
                f' | ETA={eta_min:.1f} min'
            )

    return {
        'loss': float(np.mean(losses)),
        'ctc_loss': float(np.mean(ctc_losses)),
        'anti_loss': float(np.mean(anti_losses)),
    }


@torch.no_grad()
def evaluate(model, loader):
    model.eval()

    losses = []
    refs = []
    preds = []
    blank_count = 0
    total_steps = 0

    for batch in loader:
        face = batch['face'].to(DEVICE, non_blocking=True)
        mouth = batch['mouth'].to(DEVICE, non_blocking=True)
        lengths = batch['video_lengths']
        targets = batch['targets'].to(DEVICE, non_blocking=True)
        target_lengths = batch['target_lengths']

        with torch.amp.autocast('cuda', dtype=torch.float16):
            logits = model(face, mouth, lengths)
            log_probs = logits.log_softmax(dim=-1).transpose(0, 1)
            loss = criterion(log_probs, targets, lengths, target_lengths)

        losses.append(float(loss.detach().cpu()))

        batch_preds = ctc_greedy_decode(logits.float().cpu(), lengths)
        preds.extend(batch_preds)
        refs.extend(batch['texts'])

        argmax_ids = logits.argmax(dim=-1).cpu()
        for b in range(argmax_ids.size(0)):
            T = int(lengths[b])
            seq = argmax_ids[b, :T]
            blank_count += int((seq == BLANK_ID).sum())
            total_steps += T

    cer = corpus_cer(refs, preds)
    blank_ratio = blank_count / max(total_steps, 1)

    return {
        'loss': float(np.mean(losses)),
        'cer': float(cer),
        'blank_ratio': float(blank_ratio),
        'preds': preds,
        'refs': refs,
    }

print('Train/eval functions ready')


In [ ]:
# CELL 17 — Full training 100 epochs
BEST_PATH = MODEL_ROOT / 'best_model.pt'
LAST_PATH = MODEL_ROOT / 'last_model.pt'
HISTORY_PATH = MODEL_ROOT / 'training_history.csv'

history = []
start_epoch = 1
best_cer = float('inf')
best_val_loss = float('inf')
bad_epochs = 0

if RESUME_IF_AVAILABLE and LAST_PATH.exists():
    ckpt = torch.load(LAST_PATH, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])

    if 'optimizer_state_dict' in ckpt:
        optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    if 'scaler_state_dict' in ckpt:
        scaler.load_state_dict(ckpt['scaler_state_dict'])

    start_epoch = int(ckpt.get('epoch', 0)) + 1
    best_cer = float(ckpt.get('best_cer', float('inf')))
    best_val_loss = float(ckpt.get('best_val_loss', float('inf')))
    bad_epochs = int(ckpt.get('bad_epochs', 0))

    if HISTORY_PATH.exists():
        history = pd.read_csv(HISTORY_PATH).to_dict('records')

    print('✅ Resume from epoch:', start_epoch)
else:
    print('✅ Start fresh V4')

print('='*100)
print('START DUAL-STREAM V4 TRAINING')
print('='*100)

for epoch in range(start_epoch, EPOCHS + 1):
    epoch_start = time.time()
    print()
    print(f'🚀 Epoch {epoch}/{EPOCHS}')

    train_stats = train_one_epoch(model, train_loader, epoch)
    val_stats = evaluate(model, val_loader)

    scheduler.step(val_stats['loss'])
    lr_now = optimizer.param_groups[0]['lr']
    epoch_min = (time.time() - epoch_start) / 60

    print('-'*100)
    print(
        f'Epoch {epoch:03d}/{EPOCHS} | '
        f'train={train_stats["loss"]:.4f} | '
        f'val={val_stats["loss"]:.4f} | '
        f'CER={val_stats["cer"]:.4f} | '
        f'blank={val_stats["blank_ratio"]*100:.2f}% | '
        f'lr={lr_now:.2e} | '
        f'{epoch_min:.1f} min'
    )

    for i in range(min(3, len(val_stats['refs']))):
        print('REF :', val_stats['refs'][i])
        print('PRED:', val_stats['preds'][i])

    row = {
        'epoch': epoch,
        'train_loss': train_stats['loss'],
        'train_ctc_loss': train_stats['ctc_loss'],
        'train_anti_loss': train_stats['anti_loss'],
        'val_loss': val_stats['loss'],
        'val_cer': val_stats['cer'],
        'val_blank_ratio': val_stats['blank_ratio'],
        'lr': lr_now,
        'minutes': epoch_min,
    }
    history.append(row)
    pd.DataFrame(history).to_csv(HISTORY_PATH, index=False, encoding='utf-8-sig')

    # ถือว่าดีขึ้นเมื่อ CER ลด หรือ CER เท่ากันแต่ val loss ลด
    cer_improved = val_stats['cer'] < (best_cer - 1e-6)
    tie_loss_improved = (
        abs(val_stats['cer'] - best_cer) <= 1e-6
        and val_stats['loss'] < best_val_loss
    )
    improved = cer_improved or tie_loss_improved

    if improved:
        best_cer = val_stats['cer']
        best_val_loss = val_stats['loss']
        bad_epochs = 0

        torch.save({
            'version': 'v4_stable_dualstream',
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scaler_state_dict': scaler.state_dict(),
            'best_cer': best_cer,
            'best_val_loss': best_val_loss,
            'val_blank_ratio': val_stats['blank_ratio'],
            'vocab_size': VOCAB_SIZE,
            'itos': itos,
        }, BEST_PATH)
        print('✅ Saved BEST:', BEST_PATH)
    else:
        bad_epochs += 1

    torch.save({
        'version': 'v4_stable_dualstream',
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scaler_state_dict': scaler.state_dict(),
        'best_cer': best_cer,
        'best_val_loss': best_val_loss,
        'bad_epochs': bad_epochs,
        'vocab_size': VOCAB_SIZE,
        'itos': itos,
    }, LAST_PATH)

    print(f'bad_epochs={bad_epochs}/{PATIENCE}')

    # guardrail: ถ้ายัง collapse หนักมากหลัง 10 epochs ให้เตือน ไม่หยุดอัตโนมัติ
    if epoch == 10 and val_stats['cer'] >= 0.999 and val_stats['blank_ratio'] >= 0.99:
        print()
        print('⚠️ WARNING: Epoch 10 ยัง CER≈1.0 และ blank≈100%')
        print('ควรหยุดด้วยมือและตรวจ data/alignment ก่อนปล่อยถึง 100 epochs')

    if epoch >= MIN_EPOCHS_BEFORE_EARLY_STOP and bad_epochs >= PATIENCE:
        print()
        print(f'⏹ Early stopping at epoch {epoch}')
        break

print()
print('TRAINING FINISHED')


In [ ]:
# CELL 18 — Final test ด้วย BEST checkpoint
if not BEST_PATH.exists():
    raise FileNotFoundError(f'ไม่พบ best checkpoint: {BEST_PATH}')

ckpt = torch.load(BEST_PATH, map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])

test_stats = evaluate(model, test_loader)

print('='*100)
print('FINAL TEST — DUAL STREAM V4')
print('='*100)
print('Best epoch :', ckpt.get('epoch'))
print(f'Test Loss  : {test_stats["loss"]:.4f}')
print(f'Test CER   : {test_stats["cer"]:.4f}')
print(f'Blank ratio: {test_stats["blank_ratio"]:.4f} ({test_stats["blank_ratio"]*100:.2f}%)')

for i, (ref, pred) in enumerate(zip(test_stats['refs'], test_stats['preds'])):
    print()
    print(f'[{i:03d}]')
    print('REF :', ref)
    print('PRED:', pred)
    if i >= 9:
        break


In [ ]:
# CELL 19 — Save config / summary
config = {
    'version': 'v4_stable_dualstream',
    'architecture': 'FaceCNN + MouthCNN + LearnedGateFusion + BiGRU + CTC',
    'face_size': FACE_SIZE,
    'mouth_size': MOUTH_SIZE,
    'max_frames': MAX_FRAMES,
    'batch_size': BATCH_SIZE,
    'grad_accum_steps': GRAD_ACCUM_STEPS,
    'feature_dim': FEATURE_DIM,
    'gru_hidden': GRU_HIDDEN,
    'gru_layers': GRU_LAYERS,
    'dropout': DROPOUT,
    'epochs': EPOCHS,
    'learning_rate': LR,
    'weight_decay': WEIGHT_DECAY,
    'anti_blank_weight': ANTI_BLANK_WEIGHT,
    'anti_blank_max_epoch': ANTI_BLANK_MAX_EPOCH,
    'anti_blank_threshold': ANTI_BLANK_THRESHOLD,
    'blank_id': BLANK_ID,
    'unk_id': UNK_ID,
    'vocab_size': VOCAB_SIZE,
    'best_selection': 'lowest CER, tie-break by val loss',
}

with open(MODEL_ROOT / 'model_config.json', 'w', encoding='utf-8') as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

print('Saved config:', MODEL_ROOT / 'model_config.json')
print('Best model  :', BEST_PATH)
print('Last model  :', LAST_PATH)
print('History     :', HISTORY_PATH)


## วิธีอ่านผลระหว่างเทรน

ให้ดู 3 ค่าเป็นหลัก:

1. **CER** — ยิ่งต่ำยิ่งดี
   - `1.0000` หมายถึง error ประมาณ 100%
   - ถ้าเริ่มลดต่ำกว่า 1 แสดงว่าโมเดลเริ่มสร้างลำดับตัวอักษรที่ใกล้ target มากขึ้น

2. **blank ratio** — ถ้าอยู่ใกล้ `100%` ตลอด แปลว่า CTC blank collapse
   - ช่วงแรกสูงได้
   - แต่ควรเริ่มลดลงเมื่อ training เดินหน้า

3. **PRED** — ต้องค่อย ๆ เปลี่ยนจากว่าง → ตัวอักษรบางตัว → คำ/วลีที่ใกล้ REF

### จุดตัดสินใจสำคัญ
- Epoch 1–3: ยังสรุปไม่ได้
- Epoch 5–10: ควรเริ่มเห็น non-blank เพิ่มขึ้น
- ถ้า Epoch 10 ยัง `CER≈1.0`, `blank>=99%`, และ PRED ว่างเกือบทั้งหมด: **หยุดก่อน ไม่ควรปล่อยถึง 100 epochs**
- ถ้า CER/blank ดีขึ้นต่อเนื่อง: ปล่อยต่อได้จน early stopping หรือครบ 100 epochs
